In [ ]:
import os
import sys
import json
import copy
import importlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
for fn in ("config.json", "context_config.json", "model_config.json"):
    assert os.path.exists(os.path.join(BASE_DIR, fn)), f"Missing {fn} -- run the earlier phases first."
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
context_config = json.load(open(os.path.join(BASE_DIR, "context_config.json")))
model_config = json.load(open(os.path.join(BASE_DIR, "model_config.json")))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
from meld_models import ContextFusionModel, FUSION_MODES, U_NORMS, compute_loss, class_weights_from_labels, load_split_dataset
from meld_context import make_loader, batch_to_device

SEED = config["seed"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = config["paths"]["results_dir"]
EMOTIONS, IN_DIMS = config["emotions"], context_config["in_dims"]
MODEL_KWARGS = model_config["model_kwargs"]
T_PASSES = MODEL_KWARGS["mc_passes"]
LAMBDA_GRID = [0.1, 1.0, 10.0]
assert T_PASSES == 10 and MODEL_KWARGS["u_norm"] == "mean"
print(f"Device: {device} | T={T_PASSES} | MC scope: {MODEL_KWARGS['mc_scope']} | u normalisation: {MODEL_KWARGS['u_norm']} | lambda grid: {LAMBDA_GRID}")

In [ ]:
def build_model(fusion, lam=1.0, seed=SEED, **overrides):
    torch.manual_seed(seed)
    return ContextFusionModel(fusion=fusion, lam=lam, **{**MODEL_KWARGS, **overrides}).to(device)

proposed = build_model("reliability_gated", lam=1.0)
baseline = build_model("gated", lam=0.0)
print(f"PROPOSED: {sum(p.numel() for p in proposed.parameters()):,} parameters (baseline: {sum(p.numel() for p in baseline.parameters()):,})")

df_train, train_ds = load_split_dataset(BASE_DIR, "train", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
df_dev, dev_ds = load_split_dataset(BASE_DIR, "dev", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
dev_loader = make_loader(dev_ds, batch_size=16, shuffle=False, seed=SEED)
calib_sub = Subset(train_ds, range(min(200, len(train_ds))))                 
calib_loader = make_loader(calib_sub, batch_size=32, shuffle=False, seed=0)
batch = next(iter(dev_loader))
mask_cpu = batch["mask"]; mask = mask_cpu.to(device); mk = mask_cpu.numpy()
MODS = proposed.modalities

cal = proposed.calibrate_uncertainty(calib_loader, device)
print("calibrated u_mu:", np.round(cal["u_mu"], 6), "| u_sigma:", np.round(cal["u_sigma"], 6), f"| n={cal['n']} utterances")

In [ ]:
results = []
def check(name, cond, detail=""):
    detail = str(detail)
    results.append({"test": name, "passed": bool(cond), "detail": detail})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

def run(model, b, train=False, **kw):
    model.train(train)
    with torch.set_grad_enabled(train):
        return model(batch_to_device(b, device), **kw)

def softmax64(x):
    x = x - x.max(-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(-1, keepdims=True)

def seeded_eval_u(model, loader, seed=0):
    """Replays the calibration pass (same seed / order) and returns u_raw, u (normalised) for all real utterances."""
    model.eval(); ur, un = [], []
    devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(seed)
        with torch.no_grad():
            for b in loader:
                o = model(batch_to_device(b, device))
                m_ = b["mask"].to(device)
                ur.append(o["u_raw"][m_].double().cpu().numpy()); un.append(o["u"][m_].double().cpu().numpy())
    return np.concatenate(ur), np.concatenate(un)

proposed.eval()
torch.manual_seed(0)
out = run(proposed, batch, return_mc=True)

mc = out["mc_probs"]
check("U1 T=10 MC passes -> valid distributions with non-zero variance (eval mode)",
      all(mc[m].shape[0] == T_PASSES and mc[m].shape[-1] == 7 for m in MODS)
      and all(torch.allclose(mc[m].sum(-1), torch.ones_like(mc[m].sum(-1)), atol=1e-5) for m in MODS)
      and all((mc[m].var(0, unbiased=False)[mask].mean() > 0).item() for m in MODS))

u_raw = out["u_raw"].detach().cpu().numpy()
worst = max(np.abs(mc[m].detach().cpu().numpy().astype(np.float64).var(axis=0, ddof=0).mean(-1) - u_raw[..., i]).max() for i, m in enumerate(MODS))
check("U2 raw uncertainty matches independent NumPy formula", worst < 1e-6, f"max |diff|={worst:.1e}")
check("U2b raw u is in [0, 0.25]", (u_raw[mk] >= 0).all() and (u_raw[mk] <= 0.25).all(), f"min={u_raw[mk].min():.2e}, max={u_raw[mk].max():.2e}")

ur_all, _ = seeded_eval_u(proposed, calib_loader)
mu_ref, sd_ref = ur_all.mean(0), ur_all.std(0)
check("N1 calibrated mu/sigma equal independent NumPy mean/std of u_raw on the calibration set",
      np.allclose(proposed.u_mu.cpu().numpy(), mu_ref, rtol=1e-4) and np.allclose(proposed.u_sigma.cpu().numpy(), sd_ref, rtol=1e-3),
      f"mu={np.round(mu_ref, 6)}")

_, un_all = seeded_eval_u(proposed, calib_loader)
check("N2 'mean' normalisation: mean of u over the calibration set = 1 for every modality", np.allclose(un_all.mean(0), 1.0, atol=1e-4), np.round(un_all.mean(0), 5))

z = build_model("reliability_gated", lam=1.0, u_norm="zscore"); z.load_state_dict(proposed.state_dict())
_, uz = seeded_eval_u(z, calib_loader)
check("N3 'zscore' normalisation: mean 0, std 1 on the calibration set", np.allclose(uz.mean(0), 0, atol=1e-3) and np.allclose(uz.std(0), 1, atol=1e-3),
      f"mean={np.round(uz.mean(0), 4)}, std={np.round(uz.std(0), 4)}")

worst = 0.0
for lam in LAMBDA_GRID:
    proposed.lam = lam
    o = run(proposed, batch)
    s_ = o["gate_scores"].detach().cpu().numpy().astype(np.float64)
    u_ref = o["u_raw"].detach().cpu().numpy().astype(np.float64) / proposed.u_mu.cpu().numpy().astype(np.float64)
    worst = max(worst, np.abs(o["w"].detach().cpu().numpy() - softmax64(s_ - lam * u_ref))[mk].max())
proposed.lam = 1.0
check("N4 w = softmax(s - lambda * u_raw/mu) for lambda in {0.1, 1, 10}", worst < 1e-6, f"max |diff|={worst:.1e}")

u_t = run(proposed, batch)["u"].detach()
s = run(proposed, batch)["gate_scores"].detach().cpu().numpy().astype(np.float64)
worst_grid, worst_extreme = 0.0, 0.0
for lam in [0.0] + LAMBDA_GRID + [100.0]:
    proposed.lam = lam
    w_model = run(proposed, batch, u_override=u_t)["w"].detach().cpu().numpy()
    d = np.abs(w_model - softmax64(s - lam * u_t.cpu().numpy().astype(np.float64)))[mk].max()
    if lam <= 10.0:
        worst_grid = max(worst_grid, d)
    else:
        worst_extreme = d                      
proposed.lam = 1.0
check("U3 w = softmax(s - lambda*u) for lambda in {0, 0.1, 1, 10} (1e-6) and 100 (float32: 1e-3)",
      worst_grid < 1e-6 and worst_extreme < 1e-3, f"max |diff| grid={worst_grid:.1e}, lambda=100: {worst_extreme:.1e}")

baseline.load_state_dict(proposed.state_dict()); baseline.eval()
proposed.lam = 0.0
o_p, o_b = run(proposed, batch), run(baseline, batch)
proposed.lam = 1.0
check("U4 proposed with lambda=0 reproduces the baseline exactly", torch.allclose(o_p["logits"], o_b["logits"], atol=1e-6) and torch.allclose(o_p["w"], o_b["w"], atol=1e-7))

rng = np.random.RandomState(0)
u_fake = torch.tensor(rng.uniform(0, 3.0, size=out["u"].shape), dtype=torch.float32, device=device)
proposed.lam = 2.0
o5 = run(proposed, batch, u_override=u_fake)
proposed.lam = 1.0
w5, s5, uf = (t.detach().cpu().numpy().astype(np.float64) for t in (o5["w"], o5["gate_scores"], u_fake))
lhs = np.log(w5[..., 0] / w5[..., 1])[mk]
rhs = ((s5[..., 0] - s5[..., 1]) - 2.0 * (uf[..., 0] - uf[..., 1]))[mk]
check("U5 log(w_a/w_b) = (s_a - s_b) - lambda (u_a - u_b)", np.abs(lhs - rhs).max() < 1e-4, f"max |diff|={np.abs(lhs - rhs).max():.1e}")

u_lo, u_hi = torch.zeros_like(u_fake), torch.zeros_like(u_fake)
u_hi[..., 1] = 1.5
proposed.lam = 2.0
w_lo, w_hi = run(proposed, batch, u_override=u_lo)["w"], run(proposed, batch, u_override=u_hi)["w"]
proposed.lam = 0.0
w_hi0, w_lo0 = run(proposed, batch, u_override=u_hi)["w"], run(proposed, batch, u_override=u_lo)["w"]
proposed.lam = 1.0
check("U6 higher uncertainty lowers that modality's weight (and only if lambda>0)",
      bool((w_hi[..., 1][mask] < w_lo[..., 1][mask]).all()) and bool((w_hi[..., 0][mask] > w_lo[..., 0][mask]).all()) and torch.allclose(w_hi0, w_lo0))

proposed.train(); proposed.zero_grad()
o7 = run(proposed, batch, train=True)
o7["w"][..., 0][mask].mean().backward()
head_grads = [p.grad for n, p in proposed.named_parameters() if n.startswith("modality_heads")]
check("U7a u is detached: weights give no gradient to the modality heads", not o7["u"].requires_grad and all(g is None or g.abs().sum() == 0 for g in head_grads))
check("U7b lambda and u_mu/u_sigma are not parameters (u_mu/u_sigma are saved buffers)",
      not any(("lam" in n) or ("u_mu" in n) or ("u_sigma" in n) for n, _ in proposed.named_parameters())
      and "u_mu" in proposed.state_dict() and "u_sigma" in proposed.state_dict())
proposed.zero_grad(); proposed.eval()

check("U8a proposed model computes u in training mode", run(proposed, batch, train=True)["u"] is not None)
proposed.eval()
torch.manual_seed(5); a = run(proposed, batch)["u"]
torch.manual_seed(5); b_ = run(proposed, batch)["u"]
check("U8b seeded MC passes are reproducible", torch.equal(a, b_))

pure = build_model("reliability_only", lam=2.0).eval(); pure.load_state_dict(proposed.state_dict())
o9 = run(pure, batch, u_override=u_t)
check("U9 reliability_only equals softmax(-lambda*u)", np.abs(o9["w"].detach().cpu().numpy() - softmax64(-2.0 * u_t.cpu().numpy().astype(np.float64)))[mk].max() < 1e-6)

full = build_model("reliability_gated", lam=1.0, mc_scope="full").eval(); full.load_state_dict(proposed.state_dict())
torch.manual_seed(0); o10 = run(full, batch)
u_full = o10["u_raw"].detach().cpu().numpy()
check("U10 mc_scope='full' gives valid u_raw that differs from 'head' scope",
      np.isfinite(u_full).all() and (u_full[mk] >= 0).all() and (u_full[mk] <= 0.25).all() and np.abs(u_full - u_raw).max() > 1e-8,
      f"mean u_raw: head={u_raw[mk].mean():.2e}, full={u_full[mk].mean():.2e}")

def sigmoid64(x):
    return 1.0 / (1.0 + np.exp(-x))
def logsoftmax64(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

pg = build_model("poe_gated", lam=0.0).eval(); pg.load_state_dict(proposed.state_dict())
o = run(pg, batch)
v_np = np.stack([batch["valid"][m].numpy() for m in MODS], -1).astype(np.float64)
s_np = o["gate_scores"].detach().cpu().numpy().astype(np.float64)
logp_np = np.stack([logsoftmax64(o["modality_logits"][m].detach().cpu().numpy().astype(np.float64)) for m in MODS], -2)
z_ref = ((sigmoid64(s_np) * v_np)[..., None] * logp_np).sum(-2)
check("P1 poe_gated: logits = sum_m valid_m * sigmoid(s_m) * log_softmax(head_m)", np.abs(o["logits"].detach().cpu().numpy() - z_ref)[mk].max() < 1e-4,
      f"max |diff|={np.abs(o['logits'].detach().cpu().numpy() - z_ref)[mk].max():.1e}")

u_fake2 = torch.tensor(np.random.RandomState(3).uniform(0, 3.0, size=out["u"].shape), dtype=torch.float32, device=device)
for hinge in (False, True):
    pr = build_model("poe_reliability", lam=1.5, u_hinge=hinge).eval(); pr.load_state_dict(proposed.state_dict())
    o = run(pr, batch, u_override=u_fake2)
    ue = np.maximum(u_fake2.cpu().numpy().astype(np.float64) - 1.0, 0) if hinge else u_fake2.cpu().numpy().astype(np.float64)
    g_ref = sigmoid64(o["gate_scores"].detach().cpu().numpy().astype(np.float64)) * np.exp(-1.5 * ue) * v_np
    check(f"P2 poe_reliability gates = valid * sigmoid(s) * exp(-lambda * u{'_hinge' if hinge else ''})", np.abs(o["gates"].detach().cpu().numpy() - g_ref)[mk].max() < 1e-5,
          f"max |diff|={np.abs(o['gates'].detach().cpu().numpy() - g_ref)[mk].max():.1e}")
    if hinge:
        sg = sigmoid64(o["gate_scores"].detach().cpu().numpy().astype(np.float64)) * v_np
        check("P2b with the hinge, reliability can only ATTENUATE the content gate (gate <= sigmoid(s)); equal where u <= 1",
              bool((o["gates"].detach().cpu().numpy() <= sg + 1e-6).all()) and np.allclose(o["gates"].detach().cpu().numpy()[ue == 0], sg[ue == 0], atol=1e-6))

bm = copy.deepcopy(batch); bm["valid"]["text"][:] = False
o_a = run(pg, bm)
bm2 = copy.deepcopy(bm); bm2["features"]["text"] = bm2["features"]["text"] + 50.0
o_b = run(pg, bm2)
check("P3 masked modality: gate exactly 0 and output independent of its features",
      float(o_a["gates"][..., 0][mask].abs().max()) == 0.0 and torch.allclose(o_a["logits"], o_b["logits"], atol=1e-5), f"max |dlogit|={float((o_a['logits'] - o_b['logits']).abs().max()):.1e}")
bm3 = copy.deepcopy(batch)
for m_ in MODS:
    bm3["valid"][m_][:] = False
o_c = run(pg, bm3)
check("P3b all modalities unusable -> finite (uniform) prediction, no NaN", bool(torch.isfinite(o_c["logits"]).all()) and bool(torch.isfinite(o_c["w"]).all()))

pr0 = build_model("poe_reliability", lam=0.0).eval(); pr0.load_state_dict(proposed.state_dict())
check("P4 poe_reliability with lambda=0 reproduces poe_gated exactly", torch.allclose(run(pr0, batch)["logits"], run(pg, batch)["logits"], atol=1e-6))

pm_ = build_model("poe_reliability", lam=1.0, u_measure="mi").eval(); pm_.load_state_dict(proposed.state_dict())
torch.manual_seed(0); o5m = run(pm_, batch, return_mc=True)
worst = 0.0
for i, m_ in enumerate(MODS):
    P = o5m["mc_probs"][m_].detach().cpu().numpy().astype(np.float64)
    pbar = P.mean(0)
    mi = -(pbar * np.log(pbar + 1e-12)).sum(-1) + (P * np.log(P + 1e-12)).sum(-1).mean(0)
    worst = max(worst, np.abs(np.maximum(mi, 0) - o5m["u_raw"].detach().cpu().numpy().astype(np.float64)[..., i])[mk].max())
check("P5 u_measure='mi' equals independent NumPy mutual information (>= 0)", worst < 1e-6, f"max |diff|={worst:.1e}")

check("P6 poe_* models have the same parameter count as the legacy baseline", sum(p.numel() for p in pg.parameters()) == sum(p.numel() for p in baseline.parameters()))

fresh = build_model("reliability_gated", lam=1.0); fresh.load_state_dict(proposed.state_dict()); fresh.eval()
torch.manual_seed(3); a = run(proposed.eval(), batch)["w"]
torch.manual_seed(3); b_ = run(fresh, batch)["w"]
base_cal = build_model("gated", lam=0.0); cal_b = base_cal.calibrate_uncertainty(calib_loader, device)
check("N5a state_dict round trip preserves u statistics and outputs", torch.equal(fresh.u_mu, proposed.u_mu) and torch.equal(a, b_))
check("N5b baseline calibrates its u statistics too (same training-set procedure)", all(v > 0 for v in cal_b["u_mu"]) and cal_b["u_mu"] != [1.0] * 3)
ob = run(base_cal.eval(), batch)
check("N5c baseline weights remain content-only regardless of normalisation", torch.equal(ob["w"], ob["w_content"]))

failed = [r for r in results if not r["passed"]]
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed so far")

In [ ]:
_, _ = None, None
full_calib = make_loader(train_ds, batch_size=32, shuffle=False, seed=0)
proposed.calibrate_uncertainty(full_calib, device)                       
diag = {"calibration": {"u_mu": proposed.u_mu.tolist(), "u_sigma": proposed.u_sigma.tolist(), "n_train_utterances": len(train_ds.labels)}, "tv_shift": {}}
state = proposed.state_dict()
for norm in U_NORMS:
    m_ = build_model("reliability_gated", lam=1.0, u_norm=norm); m_.load_state_dict(state); m_.eval()
    diag["tv_shift"][norm] = {}
    for lam in LAMBDA_GRID:
        m_.lam = lam
        tv, n = 0.0, 0
        torch.manual_seed(0)
        for bt in dev_loader:
            o = run(m_, bt)
            sel = bt["mask"].to(device)
            tv += (0.5 * (o["w"] - o["w_content"]).abs().sum(-1))[sel].sum().item(); n += int(sel.sum())
        diag["tv_shift"][norm][str(lam)] = tv / n
print("Mean total-variation shift of the fusion weights away from the content gate (dev, untrained model):")
print(pd.DataFrame(diag["tv_shift"]).T.to_string(float_format=lambda x: f"{x:.5f}"))
none10, mean10 = diag["tv_shift"]["none"]["10.0"], diag["tv_shift"]["mean"]["10.0"]
check("N6 normalisation makes the reliability term active (shift at lambda=10 >> unnormalised, and monotone in lambda)",
      mean10 > 0.05 and mean10 > 10 * none10 and diag["tv_shift"]["mean"]["0.1"] < diag["tv_shift"]["mean"]["1.0"] < mean10,
      f"mean-normalised {mean10:.3f} vs unnormalised {none10:.5f}")
with open(os.path.join(RESULTS_DIR, "reliability_scale_diagnostics.json"), "w") as f:
    json.dump(diag, f, indent=2)

failed = [r for r in results if not r["passed"]]
with open(os.path.join(RESULTS_DIR, "proposed_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(results)} verification checks passed.")

In [ ]:
proposed_summary = {"model_name": "ContextFusionModel(fusion='reliability_gated')",
                    "parameters": sum(p.numel() for p in proposed.parameters()),
                    "mc_passes": T_PASSES, "mc_scope": MODEL_KWARGS["mc_scope"], "lambda_grid": LAMBDA_GRID, "lambda_trainable": False,
                    "uncertainty_raw": "mean_c Var_t P_t(y=c|x_m), population variance, detached",
                    "uncertainty_normalisation": MODEL_KWARGS["u_norm"] + " (train-set statistic per modality, refreshed before training and after every epoch)",
                    "weights": "softmax(content_score - lambda * u_normalised)",
                    "poe_family": "tempered product of experts: logits = sum_m valid_m * sigmoid(s_m) * exp(-lambda*u_m) * log_softmax(head_m)  (opt-in via MELD_P9_FAMILY=poe)",
                    "status": "ready_for_training"}
with open(os.path.join(RESULTS_DIR, "proposed_model_summary.json"), "w") as f:
    json.dump(proposed_summary, f, indent=4)
print(proposed_summary)
print("Phase 8 Proposed Model completed successfully.")